<a href="https://colab.research.google.com/github/Frabbandera/medleydb-mert-instrument-classification/blob/main/notebooks/pipeline_medleydb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/filippolonghi/medleydb-mert-instrument-classification/blob/main/notebooks/pipeline_medleydb.ipynb)


## What you can change

- `PROJECT_ROOT`, `RUN_ROOT`, and `MEDLEYDB_ROOT` for local vs Colab/Drive paths.
- Config path variables such as `EXPERIMENT_CONFIG` or `MIXTURE_CONFIG`.
- `debug` is for quick smoke tests; `largest_balanced_medleydb_instrument` is the final main dataset.
- Polyphonic modes: `synthetic_random` is controlled overlap, `same_song_reconstructed` is realistic stem reconstruction, and `original_full_mix` is real mix evaluation.


# MedleyDB frozen-MERT pipeline

This notebook runs the complete frozen-MERT isolated-stem pipeline on Google Colab: index stems, select a leakage-free balanced subset, extract frozen MERT embeddings, train the classifier head, evaluate it, and plot the results.

**Before starting:** select **Runtime -> Change runtime type -> GPU**. A Colab T4 is appropriate for the default frozen-MERT experiment because MERT is used only once during embedding extraction; the classifier training is lightweight. You also need access to the team's shared Google Drive folder and the private GitHub repository. Your counts and metrics may differ if your Drive copy of MedleyDB differs.

The repository is cloned into temporary `/content` storage. MedleyDB lives in Google Drive, and generated artifacts are synchronized back to the shared Drive folder so the same cache, checkpoints, and result tables can be reused in later Colab sessions.

## 1. Mount the shared project folder

The expected Drive layout is `MyDrive/medleydb-mert-instrument-classification/MedleyDB`. If the folder was shared with you, add a shortcut to it under **My Drive** using that exact name. The artifact folder stores embeddings, checkpoints, reports, results, and the Hugging Face cache between Colab sessions.

### When to rerun expensive stages

Use the following rule of thumb before running this notebook:

- **First full run, or after replacing a debug cache:** keep `OVERWRITE_EMBEDDINGS = True` so Stage 3 writes a clean full cache to Drive.
- **Same dataset, same subset settings, same MERT model/layer/pooling:** set `OVERWRITE_EMBEDDINGS = False`; Stage 3 can reuse the restored cache and the run is much faster.
- **Changed MedleyDB files, subset size, segment length, selected classes, MERT layer, pooling, or model name:** rerun the affected stages and create/overwrite the corresponding cache.
- **New classifier experiment on the same cache:** change only `EXPERIMENT_ID`; do not overwrite embeddings.
- **Replacing an existing result:** set `REPLACE_EXISTING = True` only when the existing result directory and registry row should intentionally be replaced.

The notebook restores saved artifacts from Drive near the beginning and syncs new artifacts back after the expensive stages.

In [ ]:
import os
import sys
from pathlib import Path

# Shared team paths on Google Drive. Keep official isolated-stem outputs here;
# do not mix old debug runs into this folder.
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/content/medleydb-mert-instrument-classification"))
MEDLEYDB_ROOT = Path(os.environ.get("MEDLEYDB_ROOT", "/content/drive/MyDrive/medleydb_mert_project/MedleyDB"))
RUN_ROOT = Path(os.environ.get("RUN_ROOT", "/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1"))
if PROJECT_ROOT.exists():
    os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from google.colab import drive
drive.mount("/content/drive")

PERSIST_ROOT = RUN_ROOT
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["HF_HOME"] = str(PERSIST_ROOT / "huggingface")

# Default real T4 experiment.
#
# Before running the expensive cells, check these values:
# 1) EXPERIMENT_ID: use a new name for a new run, for example by adding your initials.
# 2) REPLACE_EXISTING: keep False unless the team agreed to overwrite that run.
# 3) OVERWRITE_EMBEDDINGS: use True only when creating/replacing the embedding cache.
# 4) MERT_LAYER and MERT_POOLING: these must match EXPERIMENT_CONFIG.
#
# After a successful full run has been saved to Drive, set OVERWRITE_EMBEDDINGS
# to False for repeated runs with the same subset/model/layer/pooling. This
# avoids recomputing MERT embeddings and makes the pipeline much faster.
SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RUN_PROFILE = "largest_balanced_medleydb_t4"
EXPERIMENT_CONFIG = "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml"
EXPERIMENT_ID = "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_colab"
REPLACE_EXISTING = False
OVERWRITE_EMBEDDINGS = True

MODEL_NAME = "m-a-p/MERT-v1-95M"
MODEL_CACHE_NAME = "mert_v1_95m"
MERT_LAYER = "last"
MERT_POOLING = "mean"

DATASET_CONFIG = "configs/datasets/subset_largest_balanced_medleydb_instrument.yaml"
SUBSET_CSV = RUN_ROOT / "data" / "metadata" / f"subset_{SUBSET_PROFILE}_{LABEL_GRANULARITY}.csv"
LABEL_TO_ID = RUN_ROOT / "data" / "metadata" / f"labels_{SUBSET_PROFILE}_{LABEL_GRANULARITY}_label_to_id.json"
EMBEDDING_CACHE_DIR = (
    RUN_ROOT / "data" / "cache" / MODEL_CACHE_NAME / SUBSET_PROFILE / LABEL_GRANULARITY
    / f"layer_{MERT_LAYER}_pool_{MERT_POOLING}"
)

print("Drive mounted. Dataset path:", MEDLEYDB_ROOT)
print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("Subset profile:", SUBSET_PROFILE)
print("Label granularity:", LABEL_GRANULARITY)
print("Dataset config:", DATASET_CONFIG)
print("Embedding cache directory:", EMBEDDING_CACHE_DIR)
print("Official isolated-stem artifacts are saved under RUN_ROOT; keep old debug runs separate.")


In [ ]:
# 1. Search for correct folder name

!ls -la /content/drive/MyDrive

In [ ]:
# 2. Inspect two possible project folders

from pathlib import Path

candidate_roots = [
    Path("/content/drive/MyDrive/medleydb_mert_project"),
    Path("/content/drive/MyDrive/medleydb-mert-instrument-classification"),
]

for root in candidate_roots:
    print("\n" + "=" * 100)
    print("ROOT:", root)
    print("exists:", root.exists())
    print("=" * 100)

    if root.exists():
        for p in root.iterdir():
            print(p)

In [ ]:
# 3. Run correct setup

from pathlib import Path
import os
import sys

# Repository location in temporary Colab storage
PROJECT_ROOT = Path("/content/medleydb-mert-instrument-classification")

# Real MedleyDB dataset location: old shared shortcut
MEDLEYDB_ROOT = Path("/content/drive/MyDrive/medleydb-mert-instrument-classification/MedleyDB")

# Clean final run location: new project folder
RUN_ROOT = Path("/content/drive/MyDrive/medleydb_mert_project/isolated_stem_v1")
PERSIST_ROOT = RUN_ROOT

# Make sure the clean run folder exists
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Export paths so repository scripts/configs resolve correctly
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["MEDLEYDB_ROOT"] = str(MEDLEYDB_ROOT)
os.environ["RUN_ROOT"] = str(RUN_ROOT)
os.environ["HF_HOME"] = str(PERSIST_ROOT / "huggingface")

# Keep Python path ready for later, after clone
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Recompute official final-protocol paths
SUBSET_PROFILE = "largest_balanced"
LABEL_GRANULARITY = "medleydb_instrument"
RUN_PROFILE = "largest_balanced_medleydb_t4"

DATASET_CONFIG = "configs/datasets/subset_largest_balanced_medleydb_instrument.yaml"
EXPERIMENT_CONFIG = "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml"
EXPERIMENT_ID = "isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02_colab"

REPLACE_EXISTING = False
OVERWRITE_EMBEDDINGS = True

MODEL_NAME = "m-a-p/MERT-v1-95M"
MODEL_CACHE_NAME = "mert_v1_95m"
MERT_LAYER = "last"
MERT_POOLING = "mean"

SUBSET_CSV = RUN_ROOT / "data" / "metadata" / f"subset_{SUBSET_PROFILE}_{LABEL_GRANULARITY}.csv"
LABEL_TO_ID = RUN_ROOT / "data" / "metadata" / f"labels_{SUBSET_PROFILE}_{LABEL_GRANULARITY}_label_to_id.json"
EMBEDDING_CACHE_DIR = (
    RUN_ROOT / "data" / "cache" / MODEL_CACHE_NAME / SUBSET_PROFILE / LABEL_GRANULARITY
    / f"layer_{MERT_LAYER}_pool_{MERT_POOLING}"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())

print("MEDLEYDB_ROOT:", MEDLEYDB_ROOT)
print("MEDLEYDB_ROOT exists:", MEDLEYDB_ROOT.is_dir())
print("MEDLEYDB_ROOT/Audio exists:", (MEDLEYDB_ROOT / "Audio").is_dir())

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", RUN_ROOT.is_dir())

print("SUBSET_CSV:", SUBSET_CSV)
print("LABEL_TO_ID:", LABEL_TO_ID)
print("EMBEDDING_CACHE_DIR:", EMBEDDING_CACHE_DIR)

print("Environment MEDLEYDB_ROOT:", os.environ["MEDLEYDB_ROOT"])
print("Environment RUN_ROOT:", os.environ["RUN_ROOT"])

## 2. Clone the private repository and install dependencies

This cell safely clones a fresh checkout or updates an existing active checkout. Before running it, create your own GitHub personal access token with access to this private repository. In Colab, open the **Secrets** panel (key icon), add it as `GITHUB_TOKEN`, and enable notebook access. Never paste a token into the notebook or share it with another teammate.

Authentication exists only in the Git child-process environment; it is not placed in the remote URL or stored in `.git/config`.

In [ ]:
from google.colab import userdata
from pathlib import Path
import base64
import os
import subprocess

token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('GITHUB_TOKEN is unavailable in Colab Secrets')
username = "filippoLonghi"
repo_name = "medleydb-mert-instrument-classification"

REPO_URL = f"https://github.com/{username}/{repo_name}.git"
REPO_DIR = str(PROJECT_ROOT)

# Pass the token only through the child process environment. It is not
# printed, placed in the URL, or stored in the repository configuration.
authorization = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ.copy()
git_env['GIT_CONFIG_COUNT'] = '1'
git_env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
git_env['GIT_CONFIG_VALUE_0'] = f'AUTHORIZATION: basic {authorization}'

if (Path(REPO_DIR) / '.git').is_dir():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], env=git_env, check=True)
del token, authorization
git_env.pop('GIT_CONFIG_VALUE_0', None)
os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Repository ready:", Path.cwd())

In [ ]:
# 1. Verification cell

from pathlib import Path
import subprocess
import os

print("Current directory:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_ROOT exists:", Path(PROJECT_ROOT).exists())

print("\nGit status:")
subprocess.run(["git", "status", "--short"], check=True)

print("\nGit branch:")
subprocess.run(["git", "branch", "--show-current"], check=True)

print("\nLatest commit:")
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

required_files = [
    "configs/datasets/subset_largest_balanced_medleydb_instrument.yaml",
    "configs/experiments/isolated_largest_balanced_medleydb_mert95_last_mean_mlp_h256_d02.yaml",
    "src/data/build_stem_index.py",
    "src/data/create_balanced_subset.py",
    "src/features/extract_mert_embeddings.py",
    "src/experiments/run_experiment.py",
]

print("\nRequired files:")
for f in required_files:
    print(f, "->", Path(f).is_file())

print("\nEnvironment paths:")
print("MEDLEYDB_ROOT:", os.environ.get("MEDLEYDB_ROOT"))
print("RUN_ROOT:", os.environ.get("RUN_ROOT"))

### Restore and save shared artifacts

The repository checkout is temporary, but the generated metadata, MERT caches, checkpoints, and results should persist in Drive. This cell restores any previous artifacts from the shared folder and defines a helper that copies new outputs back to Drive. The helper is called after the expensive stages.

In [ ]:
# 1. Inspect RUN_ROOT folder content

from pathlib import Path

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", Path(RUN_ROOT).is_dir())

if Path(RUN_ROOT).exists():
    print("\nTop-level content of RUN_ROOT:")
    for p in sorted(Path(RUN_ROOT).iterdir()):
        if p.is_dir():
            n_files = sum(1 for x in p.rglob("*") if x.is_file())
            print(f"[DIR]  {p.name}  files={n_files}")
        else:
            print(f"[FILE] {p.name}")
else:
    print("RUN_ROOT does not exist.")

In [ ]:
from pathlib import Path
import subprocess

ARTIFACT_SUBDIRS = [
    "data/metadata",
    "data/reports",
    "data/cache",
    "checkpoints",
    "results",
]

def _rsync_dir(source: Path, destination: Path) -> None:
    source.mkdir(parents=True, exist_ok=True)
    destination.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "rsync", "-a", f"{source.as_posix()}/", f"{destination.as_posix()}/"
    ], check=True)

def restore_artifacts_from_drive() -> None:
    for rel in ARTIFACT_SUBDIRS:
        local = Path(rel)
        remote = Path(PERSIST_ROOT) / rel
        local.mkdir(parents=True, exist_ok=True)
        if remote.exists():
            _rsync_dir(remote, local)
            print(f"restored {rel}")

def sync_artifacts_to_drive() -> None:
    for rel in ARTIFACT_SUBDIRS:
        local = Path(rel)
        remote = Path(PERSIST_ROOT) / rel
        if local.exists():
            _rsync_dir(local, remote)
            print(f"saved {rel}")

restore_artifacts_from_drive()

In [ ]:
# 2. Verification cell

from pathlib import Path

print("Checking local artifact folders after restore...\n")

for rel in [
    "data/metadata",
    "data/reports",
    "data/cache",
    "checkpoints",
    "results",
]:
    path = Path(rel)
    files = [p for p in path.rglob("*") if p.is_file()]
    print(rel, "files:", len(files))

print("\nFunctions:")
try:
    restore_artifacts_from_drive
    print("restore_artifacts_from_drive: defined")
except NameError:
    print("restore_artifacts_from_drive: NOT defined")

try:
    sync_artifacts_to_drive
    print("sync_artifacts_to_drive: defined")
except NameError:
    print("sync_artifacts_to_drive: NOT defined")

In [ ]:
# 3. Identify ripristinated files

from pathlib import Path

print("LOCAL FILES RESTORED INSIDE REPOSITORY")
print("=" * 100)

for rel in [
    "data/metadata",
    "data/reports",
    "data/cache",
    "checkpoints",
    "results",
]:
    root = Path(rel)
    print("\n" + "=" * 100)
    print("LOCAL:", root)
    print("=" * 100)

    files = sorted([p for p in root.rglob("*") if p.is_file()])

    if not files:
        print("No files.")
    else:
        for p in files:
            print(p, "| size:", p.stat().st_size, "bytes")


print("\n\nFILES CURRENTLY INSIDE RUN_ROOT ON DRIVE")
print("=" * 100)
print("RUN_ROOT:", RUN_ROOT)

drive_files = sorted([p for p in Path(RUN_ROOT).rglob("*") if p.is_file()])

if not drive_files:
    print("No files in RUN_ROOT.")
else:
    for p in drive_files[:100]:
        print(p.relative_to(RUN_ROOT), "| size:", p.stat().st_size, "bytes")

    if len(drive_files) > 100:
        print("... more files not shown:", len(drive_files) - 100)

### Updating an active Colab checkout

When teammates push changes while this runtime is still active, rerun the secure clone/update cell above. Because the GitHub token is not stored in `.git/config`, a plain `!git pull` may fail for the private repository. The secure cell runs `git pull --ff-only` with a temporary authorization header and then reinstalls dependencies.

`--ff-only` prevents Git from silently creating a merge. Commit or discard intentional local code changes before pulling. The same secure workflow is explained in `docs/google_colab.md`.

## 3. Verify the GPU

`CUDA available` should be `True`. Use Colab's preinstalled CUDA-compatible PyTorch unless an actual version error occurs.

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Stage 1 - Build the stem index

This scans the metadata and audio headers, records missing or unreadable stems, and continues instead of crashing. Review `data/reports/data_health_report.md` and `bad_files.csv` after it finishes. The official MedleyDB code repository is not required.

In [ ]:
subprocess.run([
    "python", "-m", "src.data.build_stem_index",
    "--medleydb-root", str(MEDLEYDB_ROOT),
    "--out", str(RUN_ROOT / "data" / "metadata" / "stem_index.csv"),
    "--report-dir", str(RUN_ROOT / "data" / "reports"),
], check=True)

sync_artifacts_to_drive()

In [ ]:
# 1. Verification cell

from pathlib import Path
import pandas as pd

stem_index_path = RUN_ROOT / "data" / "metadata" / "stem_index.csv"
health_report_path = RUN_ROOT / "data" / "reports" / "data_health_report.md"
bad_files_path = RUN_ROOT / "data" / "reports" / "bad_files.csv"

print("stem_index.csv exists:", stem_index_path.is_file())
print("data_health_report.md exists:", health_report_path.is_file())
print("bad_files.csv exists:", bad_files_path.is_file())

if stem_index_path.is_file():
    stem_index = pd.read_csv(stem_index_path)
    print("\nStem index shape:", stem_index.shape)
    print("\nColumns:")
    print(list(stem_index.columns))
    display(stem_index.head())

if bad_files_path.is_file():
    bad_files = pd.read_csv(bad_files_path)
    print("\nBad files shape:", bad_files.shape)
    display(bad_files.head())

if health_report_path.is_file():
    print("\nData health report preview:")
    print(health_report_path.read_text()[:2000])

## Stage 2 - Create the balanced T4 subset

The dataset YAML is the source of truth for class counts, segment counts, split ratios, and label granularity. The split is performed by track, so segments from one song cannot leak across train, validation, and test sets.

For a tiny partial upload, switch `DATASET_CONFIG` to a debug YAML. Do not report that debug run as the real experiment.

In [ ]:
subprocess.run([
    "python", "-m", "src.data.create_balanced_subset",
    "--config", DATASET_CONFIG,
], check=True)

sync_artifacts_to_drive()

In [ ]:
# 1. Verification cell

from pathlib import Path
import pandas as pd
import json

subset_path = RUN_ROOT / "data" / "metadata" / "subset_largest_balanced_medleydb_instrument.csv"
label_to_id_path = RUN_ROOT / "data" / "metadata" / "labels_largest_balanced_medleydb_instrument_label_to_id.json"
id_to_label_path = RUN_ROOT / "data" / "metadata" / "labels_largest_balanced_medleydb_instrument_id_to_label.json"
legacy_label_to_id_path = RUN_ROOT / "data" / "metadata" / "medleydb_instrument_label_to_id.json"
report_path = RUN_ROOT / "data" / "reports" / "subset_report_largest_balanced_medleydb_instrument.md"

print("subset exists:", subset_path.is_file())
print("label_to_id exists:", label_to_id_path.is_file())
print("id_to_label exists:", id_to_label_path.is_file())
print("legacy medleydb_instrument_label_to_id exists:", legacy_label_to_id_path.is_file())
print("report exists:", report_path.is_file())

if subset_path.is_file():
    subset = pd.read_csv(subset_path)
    print("\nSubset shape:", subset.shape)
    print("\nColumns:")
    print(list(subset.columns))

    print("\nSplit counts:")
    display(subset["split"].value_counts().sort_index())

    print("\nClass counts:")
    display(subset["coarse_label"].value_counts().sort_index())

    print("\nClass x split:")
    display(pd.crosstab(subset["coarse_label"], subset["split"]))

    print("\nNumber of tracks:", subset["track_id"].nunique())
    print("Number of classes:", subset["coarse_label"].nunique())

    leakage_check = subset.groupby("track_id")["split"].nunique()
    leaking_tracks = leakage_check[leakage_check > 1]
    print("\nLeaking tracks:", len(leaking_tracks))
    if len(leaking_tracks) > 0:
        display(leaking_tracks.head())

if label_to_id_path.is_file():
    label_to_id = json.loads(label_to_id_path.read_text())
    print("\nlabel_to_id:")
    print(label_to_id)

if report_path.is_file():
    print("\nSubset report preview:")
    print(report_path.read_text()[:2500])

## Stage 3 - Extract frozen MERT embeddings

Each selected audio segment passes through `m-a-p/MERT-v1-95M` once. MERT remains frozen; the last hidden layer is mean-pooled into one 768-dimensional vector and cached by split. This is the slowest stage of the frozen pipeline.

If `OVERWRITE_EMBEDDINGS = False`, the extractor reuses a matching restored cache. If `OVERWRITE_EMBEDDINGS = True`, it replaces the cache. Use `True` for the first full run after a debug subset or after changing the subset/model/layer/pooling; otherwise use `False`.

In [ ]:
# 1. Preliminary check

from pathlib import Path

print("EMBEDDING_CACHE_DIR:")
print(EMBEDDING_CACHE_DIR)

for split in ["train", "val", "test"]:
    path = EMBEDDING_CACHE_DIR / f"{split}.pt"
    print(split, "exists:", path.is_file())

print("embedding_config.json exists:", (EMBEDDING_CACHE_DIR / "embedding_config.json").is_file())

In [ ]:
import yaml
from src.utils.paths import resolve_run_path

with Path(EXPERIMENT_CONFIG).open("r", encoding="utf-8") as handle:
    experiment_config = yaml.safe_load(handle)
config_cache_dir = resolve_run_path(experiment_config["data"]["cache_dir"])
if config_cache_dir != EMBEDDING_CACHE_DIR:
    raise ValueError(
        "The selected experiment config expects a different cache directory: "
        f"{config_cache_dir}. Update EXPERIMENT_CONFIG or MERT_LAYER/MERT_POOLING before extracting."
    )

cmd = [
    "python", "-m", "src.features.extract_mert_embeddings",
    "--segments", str(SUBSET_CSV),
    "--label-to-id", str(LABEL_TO_ID),
    "--medleydb-root", str(MEDLEYDB_ROOT),
    "--out-dir", str(EMBEDDING_CACHE_DIR),
    "--model-name", MODEL_NAME,
    "--batch-size", "1",
    "--device", "auto",
    "--layer", MERT_LAYER,
    "--pooling", MERT_POOLING,
    "--subset-profile", SUBSET_PROFILE,
    "--label-granularity", LABEL_GRANULARITY,
]
if OVERWRITE_EMBEDDINGS:
    cmd.append("--overwrite")
print("Embedding cache directory:", EMBEDDING_CACHE_DIR)
print("Training config consuming this cache:", EXPERIMENT_CONFIG)
print("If a valid matching cache already exists and OVERWRITE_EMBEDDINGS is False, extraction will skip MERT loading.")
subprocess.run(cmd, check=True)

sync_artifacts_to_drive()

In [ ]:
# 2. Cache Verification

from pathlib import Path
import torch
import json

print("EMBEDDING_CACHE_DIR:")
print(EMBEDDING_CACHE_DIR)

for split in ["train", "val", "test"]:
    path = EMBEDDING_CACHE_DIR / f"{split}.pt"
    print("\n" + "=" * 80)
    print(split, "exists:", path.is_file())

    if path.is_file():
        cache = torch.load(path, map_location="cpu")
        print("keys:", cache.keys())

        if "embeddings" in cache:
            print("embeddings shape:", cache["embeddings"].shape)
        if "labels" in cache:
            print("labels shape:", cache["labels"].shape)
            print("unique labels:", sorted(cache["labels"].unique().tolist()))

config_path = EMBEDDING_CACHE_DIR / "embedding_config.json"
print("\nembedding_config exists:", config_path.is_file())

if config_path.is_file():
    config = json.loads(config_path.read_text())
    print(json.dumps(config, indent=2)[:2000])

## Stage 4 and 5 - Train, evaluate, and register the experiment

This uses the experiment runner, which trains only the small MLP classifier on cached embeddings, evaluates the held-out test split, writes the full result artifact set, and appends one row to `results/experiment_registry.csv`. The 95M-parameter MERT backbone is not fine-tuned here.

In [ ]:
# 1. Preliminary Check

from pathlib import Path
import yaml
import os

print("EXPERIMENT_CONFIG:", EXPERIMENT_CONFIG)
print("EXPERIMENT_CONFIG exists:", Path(EXPERIMENT_CONFIG).is_file())

try:
    print("EXPERIMENT_ID:", EXPERIMENT_ID)
except NameError:
    print("EXPERIMENT_ID is not defined")

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", Path(RUN_ROOT).is_dir())

print("Expected cache:")
print(EMBEDDING_CACHE_DIR)

for split in ["train", "val", "test"]:
    print(split, "cache exists:", (EMBEDDING_CACHE_DIR / f"{split}.pt").is_file())

with Path(EXPERIMENT_CONFIG).open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("\nConfig experiment_id:", cfg.get("experiment_id"))
print("Config data section:")
print(cfg.get("data"))

In [ ]:
cmd = [
    "python", "-m", "src.experiments.run_experiment",
    "--config", EXPERIMENT_CONFIG,
    "--experiment-id", EXPERIMENT_ID,
]
if REPLACE_EXISTING:
    cmd.append("--replace-existing")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
sync_artifacts_to_drive()

In [ ]:
# 2. Output verification

from pathlib import Path
import pandas as pd
import json
import yaml

results_root = RUN_ROOT / "results"
checkpoints_root = RUN_ROOT / "checkpoints"

print("Results root:", results_root)
print("Results root exists:", results_root.is_dir())

print("Checkpoints root:", checkpoints_root)
print("Checkpoints root exists:", checkpoints_root.is_dir())

registry_path = results_root / "experiment_registry.csv"
print("\nexperiment_registry.csv exists:", registry_path.is_file())

if registry_path.is_file():
    registry = pd.read_csv(registry_path)
    display(registry.tail())

    if "experiment_id" in registry.columns:
        print("\nRegistered experiment IDs:")
        print(registry["experiment_id"].tolist())

# Try to locate the current experiment result directory
candidate_ids = []

try:
    candidate_ids.append(EXPERIMENT_ID)
except NameError:
    pass

with Path(EXPERIMENT_CONFIG).open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

if cfg.get("experiment_id") is not None:
    candidate_ids.append(cfg["experiment_id"])

print("\nCandidate experiment IDs:", candidate_ids)

for exp_id in candidate_ids:
    result_dir = results_root / exp_id
    checkpoint_dir = checkpoints_root / exp_id

    print("\n" + "=" * 100)
    print("Experiment:", exp_id)
    print("Result dir:", result_dir)
    print("Result dir exists:", result_dir.is_dir())
    print("Checkpoint dir:", checkpoint_dir)
    print("Checkpoint dir exists:", checkpoint_dir.is_dir())

    for fname in [
        "test_metrics.json",
        "classification_report.csv",
        "confusion_matrix_raw.csv",
        "confusion_matrix_normalized.csv",
        "confusion_matrix_raw.png",
        "confusion_matrix_normalized.png",
        "predictions.csv",
        "config_resolved.yaml",
    ]:
        print(fname, "->", (result_dir / fname).is_file())

    if (result_dir / "test_metrics.json").is_file():
        metrics = json.loads((result_dir / "test_metrics.json").read_text())
        print("\nTest metrics:")
        print(json.dumps(metrics, indent=2))



## 5. Plot and inspect the results

The cells below load the saved result files, show the main metrics, display the raw and normalized confusion matrices, and plot per-class F1. These are the main outputs used to interpret the experiment.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

result_dir = RUN_ROOT / "results" / EXPERIMENT_ID
metrics = json.loads((result_dir / "test_metrics.json").read_text())
print("Experiment:", EXPERIMENT_ID)
print(f"test accuracy:    {metrics['test_accuracy']:.4f}")
print(f"test macro-F1:    {metrics['test_macro_f1']:.4f}")
print(f"test weighted-F1: {metrics['test_weighted_f1']:.4f}")

report = pd.read_csv(result_dir / "classification_report.csv", index_col=0)
class_rows = [row for row in report.index if row not in {"accuracy", "macro avg", "weighted avg"}]
display(report.loc[class_rows + ["macro avg", "weighted avg"]])

display(Image(filename=str(result_dir / "confusion_matrix_raw.png")))
display(Image(filename=str(result_dir / "confusion_matrix_normalized.png")))

ax = report.loc[class_rows, "f1-score"].sort_values().plot.barh(figsize=(8, 4), title="Per-class F1")
ax.set_xlabel("F1 score")
plt.tight_layout()
plt.show()

predictions = pd.read_csv(result_dir / "predictions.csv")
display(predictions.head())

In [ ]:
# 1. Final sanity check

from pathlib import Path
import pandas as pd
import json

print("RUN_ROOT:", RUN_ROOT)
print("RUN_ROOT exists:", Path(RUN_ROOT).is_dir())

expected_files = {
    "stem_index": RUN_ROOT / "data" / "metadata" / "stem_index.csv",
    "subset_csv": RUN_ROOT / "data" / "metadata" / "subset_largest_balanced_medleydb_instrument.csv",
    "label_to_id": RUN_ROOT / "data" / "metadata" / "labels_largest_balanced_medleydb_instrument_label_to_id.json",
    "embedding_train": RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "train.pt",
    "embedding_val": RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "val.pt",
    "embedding_test": RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "test.pt",
    "embedding_config": RUN_ROOT / "data" / "cache" / "mert_v1_95m" / "largest_balanced" / "medleydb_instrument" / "layer_last_pool_mean" / "embedding_config.json",
    "registry": RUN_ROOT / "results" / "experiment_registry.csv",
    "test_metrics": RUN_ROOT / "results" / EXPERIMENT_ID / "test_metrics.json",
    "classification_report": RUN_ROOT / "results" / EXPERIMENT_ID / "classification_report.csv",
    "confusion_matrix_raw": RUN_ROOT / "results" / EXPERIMENT_ID / "confusion_matrix_raw.csv",
    "confusion_matrix_normalized": RUN_ROOT / "results" / EXPERIMENT_ID / "confusion_matrix_normalized.csv",
    "predictions": RUN_ROOT / "results" / EXPERIMENT_ID / "predictions.csv",
    "config_resolved": RUN_ROOT / "results" / EXPERIMENT_ID / "config_resolved.yaml",
}

print("\nExpected files:")
for name, path in expected_files.items():
    print(f"{name:28s} -> {path.is_file()}")

print("\nDataset/subset summary:")
subset = pd.read_csv(expected_files["subset_csv"])
print("subset shape:", subset.shape)
print("splits:")
print(subset["split"].value_counts().sort_index())
print("classes:", subset["coarse_label"].nunique())

print("\nPrediction summary:")
pred = pd.read_csv(expected_files["predictions"])
print("predictions shape:", pred.shape)
print("expected test rows:", len(subset[subset["split"] == "test"]))

metrics = json.loads(expected_files["test_metrics"].read_text())
print("\nTest metrics:")
print(json.dumps(metrics, indent=2))

print("\nTotal files in RUN_ROOT:")
print(sum(1 for p in Path(RUN_ROOT).rglob("*") if p.is_file()))

## 6. Optional final sync to shared Drive storage

The earlier cells already synchronize after the important stages. Run this final cell again if you manually changed or inspected files and want to be certain the shared Drive copy is up to date.

**Team note:** this is shared experiment state. Coordinate before overwriting it, or use separate experiment folders when teammates run different configurations.

In [ ]:
sync_artifacts_to_drive()
print("Shared Drive artifacts are up to date:", RUN_ROOT)